# Method 5: EfficientNet-B0 + CBAM Attention + RAF-DB
---
**Muc tieu:** Nang accuracy tu 64% (FER2013) len **85-90%** (RAF-DB + EfficientNet + CBAM)

| Thanh phan | Chi tiet |
|---|---|
| **Dataset** | RAF-DB (15,339 anh RGB 100x100, face-aligned) |
| **Backbone** | EfficientNet-B0 (pretrained ImageNet) |
| **Attention** | CBAM (Channel + Spatial) - code tu xay dung |
| **Optimizer** | AdamW + Cosine Annealing LR |
| **Training** | 2-Phase: Head -> Fine-tune Backbone |

In [ ]:
# ================================================
# CELL 1: SETUP & GPU CHECK
# ================================================
!nvidia-smi

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'GPUs: {gpus}')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU ENABLED!')
else:
    print('NO GPU! Go to Runtime > Change runtime type > GPU')

In [ ]:
# ================================================
# CELL 2: IMPORT LIBRARIES
# ================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import json
import shutil
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, Callback
)
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('Libraries imported!')

In [ ]:
# ================================================
# CELL 3: MOUNT DRIVE & COPY DATASET TO LOCAL
# ================================================
# Dataset RAF-DB da duoc upload len Google Drive duoi dang FOLDER:
#   MyDrive/CaptoneProject/dataset_raf_db/DATASET/train (7 subfolders)
#   MyDrive/CaptoneProject/dataset_raf_db/DATASET/test  (7 subfolders)
#
# Copy sang local de training NHANH hon nhieu!
# ================================================
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

DRIVE_DATASET = '/content/drive/MyDrive/CaptoneProject/dataset_raf_db'
LOCAL_PATH = '/content/dataset_rafdb'

# Tim thu muc train/test trong Drive
possible = [
    ('DATASET/train', 'DATASET/test'),
    ('train', 'test'),
]

DRIVE_TRAIN = None
DRIVE_TEST = None
for ts, es in possible:
    t = os.path.join(DRIVE_DATASET, ts)
    e = os.path.join(DRIVE_DATASET, es)
    if os.path.exists(t) and os.path.exists(e):
        DRIVE_TRAIN = t
        DRIVE_TEST = e
        print(f'Tim thay dataset tren Drive!')
        print(f'  Train: {DRIVE_TRAIN}')
        print(f'  Test:  {DRIVE_TEST}')
        break

if DRIVE_TRAIN is None:
    # Fallback: tim bat ky folder nao co 7 class
    for root, dirs, files in os.walk(DRIVE_DATASET):
        if set(['1','2','3','4','5','6','7']).issubset(set(dirs)):
            print(f'Tim thay folder chua 7 class tai: {root}')
            break
    print('Kiem tra lai cau truc folder tren Drive!')
else:
    TRAIN_DIR = os.path.join(LOCAL_PATH, 'train')
    TEST_DIR = os.path.join(LOCAL_PATH, 'test')

    if not os.path.exists(TRAIN_DIR):
        print('\nCopying dataset tu Drive sang local (1-3 phut)...')
        os.makedirs(LOCAL_PATH, exist_ok=True)
        
        print('  Copying train set...')
        shutil.copytree(DRIVE_TRAIN, TRAIN_DIR)
        tc = sum([len(f) for _, _, f in os.walk(TRAIN_DIR)])
        print(f'    -> {tc} images')
        
        print('  Copying test set...')
        shutil.copytree(DRIVE_TEST, TEST_DIR)
        ec = sum([len(f) for _, _, f in os.walk(TEST_DIR)])
        print(f'    -> {ec} images')
        print('Dataset copied!')
    else:
        tc = sum([len(f) for _, _, f in os.walk(TRAIN_DIR)])
        ec = sum([len(f) for _, _, f in os.walk(TEST_DIR)])
        print('Dataset da co san local!')

    print(f'\nTRAIN_DIR: {TRAIN_DIR} ({tc} images)')
    print(f'TEST_DIR:  {TEST_DIR} ({ec} images)')
    print(f'Classes: {sorted(os.listdir(TRAIN_DIR))}')

In [ ]:
# ================================================
# CELL 4: CONFIGURATION
# ================================================
IMG_SIZE        = 100
BATCH_SIZE      = 32
EPOCHS_PHASE1   = 15
EPOCHS_PHASE2   = 50
NUM_CLASSES     = 7
SEED            = 42
LABEL_SMOOTHING = 0.1

# flow_from_directory sorts folders alphabetically: 1,2,3,4,5,6,7
# Class 0=Folder1=Surprise, Class1=Folder2=Fear, ...
EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

np.random.seed(SEED)
tf.random.set_seed(SEED)

CHECKPOINT_DIR = '/content/drive/MyDrive/CaptoneProject/checkpoints/method5_rafdb'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_MODEL_PATH = f'{CHECKPOINT_DIR}/best_model.keras'
HISTORY_PATH = f'{CHECKPOINT_DIR}/history.pkl'

print('Config set!')
print(f'  IMG_SIZE={IMG_SIZE}, BATCH={BATCH_SIZE}')
print(f'  Phase 1: {EPOCHS_PHASE1} epochs | Phase 2: {EPOCHS_PHASE2} epochs')
print(f'  Emotions: {EMOTIONS}')

In [ ]:
# ================================================
# CELL 5: DATA GENERATORS (RGB)
# ================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest',
    validation_split=0.15
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'\nData Ready!')
print(f'  Train: {train_generator.samples} images')
print(f'  Val:   {validation_generator.samples} images')
print(f'  Test:  {test_generator.samples} images')
print(f'  Classes: {train_generator.class_indices}')

In [ ]:
# ================================================
# CELL 6: CLASS WEIGHTS & VISUALIZATION
# ================================================
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    'balanced', classes=np.unique(train_labels), y=train_labels
)
class_weights = dict(enumerate(class_weights_array))

print('Class Weights:')
for i, emotion in enumerate(EMOTIONS):
    count = np.sum(train_labels == i)
    print(f'  {emotion:12s}: {count:5d} images -> weight = {class_weights[i]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts_train = [np.sum(train_labels == i) for i in range(NUM_CLASSES)]
colors = ['#FF6B6B', '#FFE66D', '#4ECDC4', '#45B7D1', '#96CEB4', '#FF8C94', '#A8E6CF']

axes[0].barh(EMOTIONS, counts_train, color=colors)
axes[0].set_title('Train Set Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(counts_train):
    axes[0].text(v + 20, i, str(v), va='center', fontweight='bold')

test_labels_arr = test_generator.classes
counts_test = [np.sum(test_labels_arr == i) for i in range(NUM_CLASSES)]
axes[1].barh(EMOTIONS, counts_test, color=colors)
axes[1].set_title('Test Set Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(counts_test):
    axes[1].text(v + 5, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 7: VISUALIZE SAMPLE IMAGES
# ================================================
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
fig.suptitle('RAF-DB Sample Images', fontsize=16, fontweight='bold')

for class_idx in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_idx + 1))
    if os.path.exists(class_dir):
        images = sorted(os.listdir(class_dir))
        for row in range(2):
            if row < len(images):
                img = plt.imread(os.path.join(class_dir, images[row]))
                axes[row, class_idx].imshow(img)
            axes[row, class_idx].axis('off')
            if row == 0:
                axes[row, class_idx].set_title(EMOTIONS[class_idx], fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/sample_images.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 8: CBAM ATTENTION BLOCK (Code cua ban)
# ================================================
class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        ch = input_shape[-1]
        self.dense1 = layers.Dense(ch // self.ratio, activation='relu',
                                   kernel_initializer='he_normal')
        self.dense2 = layers.Dense(ch, kernel_initializer='he_normal')
        self.gap = layers.GlobalAveragePooling2D()
        self.gmp = layers.GlobalMaxPooling2D()
        super().build(input_shape)

    def call(self, x):
        ch = x.shape[-1]
        avg = self.dense2(self.dense1(self.gap(x)))
        mx  = self.dense2(self.dense1(self.gmp(x)))
        att = tf.sigmoid(avg + mx)
        return x * tf.reshape(att, (-1, 1, 1, ch))

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same',
                                  activation='sigmoid',
                                  kernel_initializer='he_normal')
        super().build(input_shape)

    def call(self, x):
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx  = tf.reduce_max(x, axis=-1, keepdims=True)
        att = self.conv(tf.concat([avg, mx], axis=-1))
        return x * att

    def get_config(self):
        config = super().get_config()
        config.update({'kernel_size': self.kernel_size})
        return config


class CBAMBlock(layers.Layer):
    def __init__(self, ratio=8, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio
        self.kernel_size = kernel_size
        self.ca = ChannelAttention(ratio=ratio)
        self.sa = SpatialAttention(kernel_size=kernel_size)

    def call(self, x):
        return self.sa(self.ca(x))

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio, 'kernel_size': self.kernel_size})
        return config

print('CBAM Attention Block defined!')

In [ ]:
# ================================================
# CELL 9: BUILD MODEL - EfficientNet-B0 + CBAM
# ================================================
def build_efficientnet_cbam(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=7):
    base_model = EfficientNetB0(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Phase 1: Freeze

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)

    # CBAM Attention
    x = CBAMBlock(ratio=16, kernel_size=7)(x)

    x = layers.GlobalAveragePooling2D()(x)

    # Classifier Head
    x = layers.Dense(512, kernel_regularizer=regularizers.l2(0.0005))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(0.0005))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model, base_model

model, base_model = build_efficientnet_cbam()
model.summary()
print(f'\nModel built! Total params: {model.count_params():,}')

In [ ]:
# ================================================
# CELL 10: PHASE 1 - TRAIN HEAD (Backbone Frozen)
# AdamW + Cosine Annealing (theo bai bao)
# ================================================
steps_per_epoch = train_generator.samples // BATCH_SIZE

cosine_p1 = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=EPOCHS_PHASE1 * steps_per_epoch,
    alpha=0.01
)

optimizer_p1 = keras.optimizers.AdamW(learning_rate=cosine_p1, weight_decay=1e-4)
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

model.compile(optimizer=optimizer_p1, loss=loss_fn, metrics=['accuracy'])

callbacks_p1 = [
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
]

print('=' * 50)
print('PHASE 1: TRAIN HEAD (Backbone Frozen)')
print(f'  AdamW + CosineDecay (1e-3 -> 1e-5)')
print(f'  Epochs: {EPOCHS_PHASE1}')
print('=' * 50)

history1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE1,
    validation_data=validation_generator,
    callbacks=callbacks_p1,
    class_weight=class_weights,
    verbose=1
)

p1_best = max(history1.history['val_accuracy'])
print(f'\nPhase 1 Done! Best Val Acc: {p1_best*100:.2f}')

In [ ]:
# ================================================
# CELL 11: PHASE 2 - FINE-TUNE BACKBONE
# ================================================
base_model.trainable = True

# QUAN TRONG: Freeze BatchNorm layers
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Unfreeze last 50 layers
FINE_TUNE_AT = len(base_model.layers) - 50
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count = sum([K.count_params(w) for w in model.trainable_weights])
print(f'Trainable params: {trainable_count:,}')

cosine_p2 = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=EPOCHS_PHASE2 * steps_per_epoch,
    alpha=0.001
)

optimizer_p2 = keras.optimizers.AdamW(learning_rate=cosine_p2, weight_decay=1e-5)
model.compile(optimizer=optimizer_p2, loss=loss_fn, metrics=['accuracy'])

callbacks_p2 = [
    ModelCheckpoint(BEST_MODEL_PATH, monitor='val_accuracy',
                    save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10,
                  restore_best_weights=True, verbose=1),
]

print('=' * 50)
print('PHASE 2: FINE-TUNE BACKBONE')
print(f'  AdamW + CosineDecay (1e-4 -> 1e-7)')
print(f'  Epochs: {EPOCHS_PHASE2} (EarlyStopping patience=10)')
print(f'  Unfrozen layers: {len(base_model.layers) - FINE_TUNE_AT}')
print('=' * 50)

history2 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE2,
    validation_data=validation_generator,
    callbacks=callbacks_p2,
    class_weight=class_weights,
    verbose=1
)

p2_best = max(history2.history['val_accuracy'])
print(f'\nPhase 2 Done! Best Val Acc: {p2_best*100:.2f}')

In [ ]:
# ================================================
# CELL 12: EVALUATE ON TEST SET
# ================================================
model = keras.models.load_model(BEST_MODEL_PATH, custom_objects={
    'CBAMBlock': CBAMBlock,
    'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention
})

test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f'\n{"="*50}')
print(f'TEST ACCURACY: {test_acc*100:.2f}')
print(f'TEST LOSS:     {test_loss:.4f}')
print(f'{"="*50}')

y_pred = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

print('\n' + classification_report(y_true, y_pred_classes, target_names=EMOTIONS))

In [ ]:
# ================================================
# CELL 13: CONFUSION MATRIX
# ================================================
cm = confusion_matrix(y_true, y_pred_classes)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax)
ax.set_title(f'Confusion Matrix - Acc: {test_acc*100:.2f}', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 14: TRAINING HISTORY
# ================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs_range = range(1, len(all_acc) + 1)
p1_end = len(history1.history['accuracy'])

axes[0].plot(epochs_range, all_acc, 'b-', label='Train', linewidth=2)
axes[0].plot(epochs_range, all_val_acc, 'r-', label='Val', linewidth=2)
axes[0].axvline(x=p1_end, color='green', linestyle='--', label='Phase 2 Start')
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, all_loss, 'b-', label='Train', linewidth=2)
axes[1].plot(epochs_range, all_val_loss, 'r-', label='Val', linewidth=2)
axes[1].axvline(x=p1_end, color='green', linestyle='--', label='Phase 2 Start')
axes[1].set_title('Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_history.png', dpi=150)
plt.show()

In [ ]:
# ================================================
# CELL 15: TEST TIME AUGMENTATION (TTA)
# ================================================
def predict_with_tta(model, test_dir, img_size, batch_size, n_aug=10):
    base_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
        test_dir, target_size=(img_size, img_size),
        color_mode='rgb', batch_size=batch_size,
        class_mode='categorical', shuffle=False
    )
    preds = model.predict(base_gen, verbose=0)

    for i in range(n_aug):
        aug_gen = ImageDataGenerator(
            rescale=1./255, rotation_range=10,
            width_shift_range=0.05, height_shift_range=0.05,
            horizontal_flip=True, zoom_range=0.1,
            brightness_range=[0.9, 1.1]
        ).flow_from_directory(
            test_dir, target_size=(img_size, img_size),
            color_mode='rgb', batch_size=batch_size,
            class_mode='categorical', shuffle=False
        )
        preds += model.predict(aug_gen, verbose=0)

    preds /= (n_aug + 1)
    return preds, base_gen.classes

print('Running TTA (10 augmentations)...')
tta_preds, y_true_tta = predict_with_tta(model, TEST_DIR, IMG_SIZE, BATCH_SIZE)
tta_classes = np.argmax(tta_preds, axis=1)
tta_acc = np.mean(tta_classes == y_true_tta)

print(f'\n{"="*50}')
print(f'Standard Accuracy: {test_acc*100:.2f}')
print(f'TTA Accuracy:      {tta_acc*100:.2f}')
print(f'Improvement:       +{(tta_acc - test_acc)*100:.2f}')
print(f'{"="*50}')

In [ ]:
# ================================================
# CELL 16: SAVE MODEL FOR WEBAPP
# ================================================
FINAL_PATH = f'{CHECKPOINT_DIR}/emotion_rafdb_final.keras'
model.save(FINAL_PATH)

WEIGHTS_PATH = f'{CHECKPOINT_DIR}/emotion_rafdb_weights.weights.h5'
model.save_weights(WEIGHTS_PATH)

config = {
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'emotions': EMOTIONS,
    'color_mode': 'rgb',
    'test_accuracy': float(test_acc),
    'tta_accuracy': float(tta_acc),
    'model_name': 'EfficientNetB0_CBAM_RAFDB'
}
with open(f'{CHECKPOINT_DIR}/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Model saved!')
print(f'  Full:    {FINAL_PATH}')
print(f'  Weights: {WEIGHTS_PATH}')
print(f'  Config:  {CHECKPOINT_DIR}/model_config.json')
print(f'\nDownload file .keras va cap nhat webapp/model.py')

In [ ]:
# ================================================
# CELL 17: FINAL SUMMARY
# ================================================
print('=' * 60)
print(' SUMMARY: Method 5 - EfficientNet-B0 + CBAM + RAF-DB')
print('=' * 60)
print(f' Dataset:       RAF-DB ({train_generator.samples + test_generator.samples} images)')
print(f' Architecture:  EfficientNet-B0 + CBAM Attention')
print(f' Optimizer:     AdamW + Cosine Annealing')
print(f' Input:         {IMG_SIZE}x{IMG_SIZE} RGB')
print(f' Params:        {model.count_params():,}')
print(f'')
print(f' Test Acc:      {test_acc*100:.2f}')
print(f' TTA Acc:       {tta_acc*100:.2f}')
print(f'')
print(f' vs Method 2 (CBAM CNN + FER2013):    64.22')
print(f' vs Method 3 (MobileNetV2 + FER2013): 36.28')
print(f' -> Improvement: +{(test_acc - 0.6422)*100:.2f}')
print('=' * 60)